# Attach FERMAT images to the case snapshots

Fifteenth notebook, and the only one here that needs a GPU-free
authenticated session rather than a GPU.

`reference/cases/` holds one bundle per qualitative case: the ground
truth, every raw model sample, the parsed values, the derived
entropy/correctness, and a human note explaining what the case shows.
Those bundles are built offline from the results CSVs
(`pilot.cases.build_case`), because the CSVs carry all the text.

What they cannot carry offline is the **image**. FERMAT is a gated
dataset, so its pages can only be fetched in a session that has HF auth --
which is this notebook's entire job. ScratchMath is ungated and its case
image is already attached locally, so it is skipped here.

This is deliberately a separate step rather than part of case
construction: a bundle without an image is still useful (the raw samples
and note are the substance), so image attachment must never be able to
block or invalidate bundle creation.

**Run this after any change to `reference/cases/`.** It is idempotent --
re-running re-attaches the same images and rewrites the same metadata.


In [ ]:
# Install + auth. No model, no GPU -- this notebook only needs HF access
# to a gated dataset and a clone of the repo.
!pip install -q datasets huggingface_hub


In [ ]:
import json
import os
from getpass import getpass

from huggingface_hub import login

from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"

TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False


def get_token(name, prompt):
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")
if not HF_TOKEN.startswith("hf_"):
    raise ValueError("Stored HF token does not start with 'hf_'; set RESET_TOKENS = True.")
login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/

import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
for _n in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_n]

import pilot.cases
import pilot.data

print("pilot imported from:", os.path.dirname(pilot.__file__))
assert hasattr(pilot.cases, "attach_image"), "stale clone -- push pilot/cases.py first"


In [ ]:
# Load the FERMAT samples the cases were drawn from. Same calls, same
# seeds as the runs themselves, so the draw is identical -- cases are
# matched back by question text rather than by row index, which survives
# any reordering.
import pilot.data

fermat_n300 = pilot.data.load_fermat_balanced(n=300, seed=42, target_error_frac=0.5)
print(f"FERMAT n=300 balanced: {len(fermat_n300)} items")

# Case 07 comes from the 7B matched-n650 run, whose extra items are drawn
# past the first 350. Loaded lazily below only if that case still needs an
# image (it is normally attached already from the notebook-07 dump).
_extra_cache = {}


def fermat_extra(skip, n_extra):
    key = (skip, n_extra)
    if key not in _extra_cache:
        _extra_cache[key] = pilot.data.load_fermat_extra_error_items(
            n_extra=n_extra, seed=42, skip=skip
        )
    return _extra_cache[key]


def build_lookup(dataset):
    """question text -> image, for matching cases back to their source row."""
    out = {}
    for item in dataset:
        out[str(item["orig_q"]).strip()] = item["image"]
    return out


lookup = build_lookup(fermat_n300)
print(f"lookup built over {len(lookup)} distinct questions")


In [ ]:
# Attach an image to every bundle that still needs one and whose question
# is findable in the FERMAT sample. Reports each case explicitly rather
# than summarising: a silently skipped case would look identical to a
# successfully attached one in the final count.
from pathlib import Path

import pilot.cases

CASES_DIR = Path("repo/reference/cases")
attached, already, unmatched = [], [], []

for meta_path in sorted(CASES_DIR.glob("*/case.json")):
    case = json.loads(meta_path.read_text())
    case_id = meta_path.parent.name

    if "image" in case:
        already.append(case_id)
        continue

    question = str(case["ground_truth"]["question"]).strip()
    image = lookup.get(question)
    if image is None:
        # Not in the n=300 draw -- either an extension item or a different
        # dataset. Widen the search once before giving up.
        widened = build_lookup(fermat_extra(skip=350, n_extra=300))
        image = widened.get(question)

    if image is None:
        unmatched.append(case_id)
        print(f"[unmatched] {case_id}: question not found in the FERMAT draw")
        continue

    path = pilot.cases.attach_image(meta_path.parent, image)
    size_kb = path.stat().st_size / 1024
    attached.append(case_id)
    print(f"[attached]  {case_id} -> {path.name} ({size_kb:.0f} KB)")

print()
print(f"attached {len(attached)}, already had images {len(already)}, unmatched {len(unmatched)}")
if unmatched:
    print("Unmatched cases keep their text bundle and simply have no image; "
          "that is a known, non-fatal state (see pilot/cases.py).")

index = pilot.cases.write_case_index(CASES_DIR)
print("wrote", index)


In [ ]:
# Display each attached image next to its note, so the bundle can be eyeballed
# here rather than only after pulling the repo.
from IPython.display import display

for meta_path in sorted(CASES_DIR.glob("*/case.json")):
    case = json.loads(meta_path.read_text())
    if "image" not in case:
        continue
    print("=" * 78)
    print(f"{meta_path.parent.name}   [{case['category']}]")
    print(f"entropy={case['derived']['entropy']:.3f}  correct={case['derived']['correct']}")
    print(f"note: {case['note'][:300]}")
    from PIL import Image
    display(Image.open(meta_path.parent / case["image"]["filename"]))


In [ ]:
# Commit the attached images back to the repo.
import subprocess

_REDACT = []


def git(*args):
    r = subprocess.run(["git", "-C", "repo", *args], capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    for s in _REDACT:
        if s:
            out = out.replace(s, "***")
    if r.returncode != 0 and out.strip():
        print(out.strip())
    return r


git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")
git("add", "reference/cases")
commit = git("commit", "-m", "Attach FERMAT images to the case snapshots")
if commit.returncode != 0:
    print("Nothing to commit (or commit failed) -- see above.")

GH_PUSH_TOKEN = (globals().get("GH_TOKEN") or "").strip()
_REDACT.append(GH_PUSH_TOKEN)
if not GH_PUSH_TOKEN:
    print("No token -- skipping push.")
else:
    push_url = REPO_URL.replace("https://", f"https://{GH_PUSH_TOKEN}@")
    if git("fetch", push_url, "main").returncode == 0:
        if git("rebase", "FETCH_HEAD").returncode != 0:
            git("rebase", "--abort")
            print("Rebase failed; attempting push anyway.")
    if git("push", push_url, "HEAD:main").returncode == 0:
        print("Pushed case images.")
    else:
        print("Push failed (see above).")
